In [1]:
!pip install -q pypdf langchain langchain-community langchain-groq sentence-transformers chromadb langchain-text-splitters

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 874.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/2

In [24]:
import os
import uuid
import torch
import pandas as pd
from tqdm.auto import tqdm
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. Setup Environment and Load Data
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={'device': device}
)

file_path = "/content/Seerat e Mustafa_new.pdf"
loader = PyPDFLoader(file_path)
documents = loader.load()
full_text = " ".join([doc.page_content for doc in documents])

# 2. Define Chunking Strategies
strategy_configs = {
    "Fixed-size": CharacterTextSplitter(separator=" ", chunk_size=500, chunk_overlap=0),
    "Overlapping": CharacterTextSplitter(separator=" ", chunk_size=500, chunk_overlap=100),
    "Recursive": RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50, separators=["\n\n", "\n", " ", ""])
}

indexed_retrievers = {}
chunk_id_maps = {}

# 3. Indexing with Unique IDs
for name, splitter in strategy_configs.items():
    docs = splitter.create_documents([full_text])
    ids = [f"{name}_{i}_{str(uuid.uuid4())[:8]}" for i in range(len(docs))]

    chunk_id_maps[name] = {}
    for doc, id_val in zip(docs, ids):
        doc.metadata["chunk_id"] = id_val
        chunk_id_maps[name][id_val] = doc.page_content

    collection_name = name.replace("-", "_").lower()
    vectorstore = Chroma.from_documents(
        documents=docs,
        embedding=embeddings,
        ids=ids,
        collection_name=collection_name
    )
    indexed_retrievers[name] = vectorstore.as_retriever(search_kwargs={"k": 5})

print("Indexing complete.")

Using device: cuda


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Indexing complete.


In [27]:
import pandas as pd

# 1. Define the 10 Evaluation Questions and their Keywords
eval_tasks = [
    {"query": "How many people participated in the Battle of Badr from the Muslim side?", "keywords": ["313", "badr", "participated"]},
    {"query": "What were the conditions of the Treaty of Hudaybiyyah?", "keywords": ["treaty", "hudaybiyyah", "conditions", "ten years"]},
    {"query": "Who was the companion that accompanied Rasulullah during the migration to Madinah?", "keywords": ["abu bakr", "migration", "cave", "thawr"]},
    {"query": "Describe the incident of the monk Bahira recognizing the Prophet.", "keywords": ["bahira", "monk", "busra", "seal"]},
    {"query": "What happened during the Conquest of Makkah regarding the idols?", "keywords": ["idols", "ka'bah", "conquest", "makkah", "broken"]},
    {"query": "Who was the first woman to embrace Islam?", "keywords": ["khadijah", "first", "woman", "embrace"]},
    {"query": "What was the reaction of the Quraish to the public invitation at Mount Safa?", "keywords": ["safa", "quraish", "abu lahab", "invitation"]},
    {"query": "Describe the physical appearance of the Prophet as mentioned by Hind ibn Abi Hala.", "keywords": ["appearance", "hind", "description", "prophet"]},
    {"query": "Who was the commander of the Muslim army at the Battle of Yarmook?", "keywords": ["yarmook", "khaalid", "commander"]},
    {"query": "What was the miracle involving the milk and the Sahaba?", "keywords": ["milk", "miracle", "vessel", "satiated"]}
]

# 2. Identify strategy-specific Golden IDs
auto_golden_map = {}
for strategy_name in strategy_configs.keys():
    auto_golden_map[strategy_name] = []
    for task in eval_tasks:
        kws = [k.lower() for k in task["keywords"]]
        valid_ids = []
        for cid, content in chunk_id_maps[strategy_name].items():
            score = sum(1 for k in kws if k in content.lower())
            if score >= 2:
                valid_ids.append(cid)

        if not valid_ids:
            best_cid = None
            max_match = -1
            for cid, content in chunk_id_maps[strategy_name].items():
                score = sum(1 for k in kws if k in content.lower())
                if score > max_match:
                    max_match = score
                    best_cid = cid
            valid_ids = [best_cid] if best_cid else []
        auto_golden_map[strategy_name].append(valid_ids)

# 3. Calculate Hit Rate@5
hit_rate_results = []
for i, task in enumerate(eval_tasks):
    q = task["query"]
    for strategy_name, retriever in indexed_retrievers.items():
        golden_ids = auto_golden_map[strategy_name][i]

        # Get top 5 results
        retrieved_docs = retriever.invoke(q)
        retrieved_ids = [d.metadata.get('chunk_id') for d in retrieved_docs]

        # A 'Hit' occurs if AT LEAST ONE golden chunk is in the retrieved set
        is_hit = any(tid in retrieved_ids for tid in golden_ids)

        hit_rate_results.append({
            "Question": q,
            "Strategy": strategy_name,
            "Hit": is_hit
        })

# 4. Display Results Table
df_hit_rate = pd.DataFrame(hit_rate_results)
print("--- Hit Rate@5 (At least one Golden Chunk found in Top 5) ---")
pivot_hit = df_hit_rate.pivot(index='Question', columns='Strategy', values='Hit')
display(pivot_hit)

print("\n--- Strategy Accuracy Summary (Hit Rate % Success) ---")
summary_hit = df_hit_rate.groupby('Strategy')['Hit'].mean() * 100
display(summary_hit)

--- Hit Rate@5 (At least one Golden Chunk found in Top 5) ---


Strategy,Fixed-size,Overlapping,Recursive
Question,,,
Describe the incident of the monk Bahira recognizing the Prophet.,True,True,True
Describe the physical appearance of the Prophet as mentioned by Hind ibn Abi Hala.,True,True,True
How many people participated in the Battle of Badr from the Muslim side?,False,False,False
What happened during the Conquest of Makkah regarding the idols?,True,True,True
What was the miracle involving the milk and the Sahaba?,False,True,False
What was the reaction of the Quraish to the public invitation at Mount Safa?,False,False,False
What were the conditions of the Treaty of Hudaybiyyah?,True,True,True
Who was the commander of the Muslim army at the Battle of Yarmook?,False,False,False
Who was the companion that accompanied Rasulullah during the migration to Madinah?,False,False,False



--- Strategy Accuracy Summary (Hit Rate % Success) ---


,Hit
Strategy,
Fixed-size,50.0
Overlapping,60.0
Recursive,50.0
